## Apresentação 

Notebook destinado ao estudo da implementação de um conversational RAG por meio do uso de uma arquitetura de grafos, fornecida pela biblioteca LangGraph, construída dentro do ecossistema do framework LangChain. Conversational RAG se trata de uma abordagem de elaboração de um chatbot que apresenta a capacidade de responder às mensagens do usuário em função de uma base de conhecimento, ao mesmo tempo em que proporciona experiência multi-turno, promovendo uma experiência realmente conversacional pela instrodução de um sistema de memória que atua para persistir as informações anteriormente realizadas entre o usuário e o modelo de LLM.

Nesse sentido, o objetivo será elaborar um chatbot que consiga promover uma experiência multi-turno, capaz de responder em função da base de conhecimento utilizada, que invariavelmente estará sujeita ao contexto de uso no qual o modelo de LLM estará inserido. 

### Library 

In [26]:
import getpass
import json
import os

# from IPython.display import Image, display
from langchain import hub
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.messages import SystemMessage
from langchain_core.tools import tool
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_groq import ChatGroq
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import Annotated, List, TypedDict

In [ ]:
# API Example: your-api-key

os.environ["GROQ_API_KEY"]=getpass.getpass("Your API Key: ")

### Testando a conexão com o modelo 

In [17]:
llm = ChatGroq(
    model = "llama3-70b-8192", 
    temperature = 0
)

In [4]:
llm.invoke("Olá, tudo bem ?").content

'Olá! Tudo bem, obrigado! E você?'

### Testando o modelo de embedding


In [3]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

c:\Users\Bruno\Documents\BrunoLod\git_repo\Language-Agents\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
vector = embeddings.embed_query(text="Solitude Noire")

vector[:5]

[0.01712828129529953,
 0.11436092853546143,
 0.008908018469810486,
 -0.0067785633727908134,
 -0.07661733776330948]

### Formação da base de conhecimento

Base de conhecimento, também conhecida como knowledge base se refere a uma fonte de informação a partir da qual o modelo utiliza para responder o usuário, visando garantir um incremento da qualidade de resposta, proporcionando uma não dependência do pré-treinamento dos modelos de LLM. 

In [37]:
"""
Elaborando os métodos utilizados para o modelo possuir
a sua base de conhecimento.  
"""

def loader(documents: Document) -> List[Document]:
    """ 
    """
    loader = PyPDFLoader(documents)
    return loader.load()

def splitter(documents: Document) -> List[Document]:
    """
    Splits the loaded documents into smaller chunks using a recursive text splitter.

    Returns:
    List[Document]: A list of document chunks.
    """
    text_splitter = RecursiveCharacterTextSplitter(
    chunk_size         = 500, 
    chunk_overlap      = 50, 
    length_function    = len,
    separators         = ["", " ", ".", "\n", "\n\n"],
    is_separator_regex = False
    )

    return text_splitter.split_documents(loader(documents=documents))    


In [38]:
vector_store = InMemoryVectorStore(
    embedding=embeddings
)

In [39]:
article_path = "Review of AI and Mental Health.pdf"
_ = vector_store.add_documents(documents=splitter(documents=article_path))


In [31]:
# Utilizando um prompt para RAG pré-pronto. 

prompt = hub.pull("rlm/rag-prompt")

c:\Users\Bruno\Documents\BrunoLod\git_repo\Language-Agents\venv\lib\site-packages\langsmith\client.py:241: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


### QA RAG - Part I

In [ ]:
class State(TypedDict):
    question: str 
    context: List[Document]
    answer: str

def retrieve(state: State) -> dict:
    """ 
    """
    retrieve_documents = vector_store.similarity_search(state["question"])
    return {"context": retrieve_documents}

def generate(state: State):
    """ 
    """
    documents_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = prompt.invoke({"question": state["question"], "context": documents_content})
    response = llm.invoke(messages)
    return {"answer": response.content}

graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()
    
# Para visualizar a relação em grafos elaboradas: 
# display(Image(graph.get_graph().draw_mermaid_png()))

### Interagindo com o modelo

In [63]:
query_1 = "Sobre o que o artigo fala ? Responda de forma breve."
query_2 = "Ele diz as principais formas de integração com psicologia e IA ? Se sim, quais ?"
query_3 = "Como elas podem ajudar na condução psicoterapêutica ?"


In [46]:
result = graph.invoke({"question": query_1})

print(f'Context: {result["context"]}')
print(f'Answer: {result["answer"]}')

Context: [Document(id='bba62a1d-94ca-41eb-8265-d8f5c997c6ab', metadata={'source': 'Review of AI and Mental Health.pdf', 'page': 8}, page_content='unced in the\nclinical/subclinical population (g = 1.069) compared to non-\nclinical population (g= 0.107, F (2,19)= 7.152, p = 0.005). Female\npercentage in the sample did not moderate the effects of CA on\npsychological distress (g = – 0.47, F (1, 19) = 0.105, p = 0.749).\nSimilarly, CA intervention effects on psychological distress did not\ndiffer by the type of control groups (F (5, 20)= 2.598, p = 0.06).\nYet, the effects of AI-based CA on psychological well-being did not\nexhibit signi ﬁcant varia'), Document(id='2122ea3f-7fe9-4c27-b938-730e6d461d2e', metadata={'source': 'Review of AI and Mental Health.pdf', 'page': 9}, page_content='agreement for title/abstract screening (0.9)\nand full-text review (0.83).\nThe full description and examples of eligibility criteria are\noutlined in Supplementary Table 8. Brie ﬂy, we developed our\neligi

In [49]:
result = graph.invoke({"question": query_2})

print(f'Context: {result["context"]}')
print(f'Answer: {result["answer"]}')

Context: [Document(id='de4ff3be-b320-4650-83d1-d194630b1306', metadata={'source': 'Review of AI and Mental Health.pdf', 'page': 5}, page_content='ed for the\ndelivery of psychotherapy and/or psychoeducational content\n(n = 22). The integrative approach and CBT emerged as the most\nprevalent therapeutic approaches, represented in 11 and 6 studies\nrespectively. Additionally, several CAs were designed to offer\nsocial assistance, companionship, or act as a source of emotional\nsupport for users18,22,30,38,40– 43. There were also instances where\nCAs were employed for speciﬁc purposes such as coaching37,44,\ncounseling45, remote monitoring 42, telec'), Document(id='961a0f73-352f-4365-b982-f581c1caac1e', metadata={'source': 'Review of AI and Mental Health.pdf', 'page': 1}, page_content='d trials, leaving 15 randomized trials eligible for meta-\nanalysis to estimate the effectiveness of AI-based CAs on\npsychological outcomes. Table 1 presents selected major\ncharacteristics of studies incl

In [51]:
result = graph.invoke({"question": query_3})

print(f'Context: {result["context"]}')
print(f'Answer: {result["answer"]}')

Context: [Document(id='de4ff3be-b320-4650-83d1-d194630b1306', metadata={'source': 'Review of AI and Mental Health.pdf', 'page': 5}, page_content='ed for the\ndelivery of psychotherapy and/or psychoeducational content\n(n = 22). The integrative approach and CBT emerged as the most\nprevalent therapeutic approaches, represented in 11 and 6 studies\nrespectively. Additionally, several CAs were designed to offer\nsocial assistance, companionship, or act as a source of emotional\nsupport for users18,22,30,38,40– 43. There were also instances where\nCAs were employed for speciﬁc purposes such as coaching37,44,\ncounseling45, remote monitoring 42, telec'), Document(id='a2594c3b-2ce1-447f-b6d8-881639861de8', metadata={'source': 'Review of AI and Mental Health.pdf', 'page': 2}, page_content='Table 1. Major characteristics of studies included in the systematic review.\nStudy and sample characteristics Intervention characteristics CA design characteristics Mechanisms Outcomes Note\nAuthor, year,\

In [66]:
"""
Testando a ausência de memória  
"""

result = graph.invoke({"question": "how was my last query ?"})

print(f'Context: {result["context"]}')
print(f'Answer: {result["answer"]}')

Context: [Document(id='86bc919c-1459-4959-a4e5-8426d163bebe', metadata={'source': 'Review of AI and Mental Health.pdf', 'page': 10}, page_content='l\nillness symptoms, or any diagnosed health issues. For the purposes of data analysis,\nwe further classiﬁed health statuses into two categories: the clinical/subclinical\npopulation and the non-clinical population.\nCA features Response generation approach:\n\x81 Retrieval-based (n = 11)\n\x81 Generative (n = 4)\nResponse generation approach pertains to the technique a CA employs to formulate\nresponses to user inputs\n\x81Retrieval-based CAs select appropriate responses from a repository of pre-existing\nconver'), Document(id='c3b3319e-285b-4cc1-b1bb-84326235f9c4', metadata={'source': 'Review of AI and Mental Health.pdf', 'page': 4}, page_content='Diabetes-5,PANAS Positive and Negative Affect Scale,PHQ-8 Patient Health Questionnaire-8,\nPHQ-9 Patient Health Questionnaire-9,PROMIS Patient-Reported Outcomes Measurement Information System,PS

### Introduzindo saída estruturada para a resposta do modelo 

In [57]:
class AnswerWithSources(TypedDict):
    """An answer to the question, with sources."""

    answer: str
    sources: Annotated[
        List[str],
        ...,
        "List of sources used to answer the question",
    ]

class State(TypedDict):
    question: str 
    context: List[Document]
    answer: AnswerWithSources

def retrieve(state: State) -> dict:
    """ 
    """
    retrieve_documents = vector_store.similarity_search(state["question"])
    return {"context": retrieve_documents}

def generate(state: State):
    """ 
    """
    documents_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = prompt.invoke({"question": state["question"], "context": documents_content})
    structured_llm = llm.with_structured_output(AnswerWithSources)
    response = structured_llm.invoke(messages)
    return {"answer": response}

graph_builder_2 = StateGraph(State).add_sequence([retrieve, generate])
graph_builder_2.add_edge(START, "retrieve")
graph_2 = graph_builder_2.compile()
    
# Para visualizar a relação em grafos elaboradas: 
# display(Image(graph.get_graph().draw_mermaid_png()))

In [64]:
result = graph_2.invoke({"question": query_2})
print(json.dumps(result["answer"], indent=2))

{
  "answer": "The integrative approach and CBT emerged as the most prevalent therapeutic approaches.",
  "sources": [
    "The studies involved 17,123 participants from 15 countries."
  ]
}


In [65]:
result = graph_2.invoke({"question": query_3})
print(json.dumps(result["answer"], indent=2))

{
  "answer": "They can help in psychotherapeutic conduction by offering social assistance, companionship, or emotional support, and by being employed for specific purposes such as coaching, counseling, or remote monitoring.",
  "sources": [
    "study 18",
    "study 22",
    "study 30",
    "study 38",
    "study 40-43"
  ]
}


### Naive Conversational RAG

In [40]:
"""
Criando métodos e trechos que irá criar 
um modelo básico como conversational RAG, 
permitindo a interação multi-turno a partir
de um checkpointer básico que irá adicionar
cada interação numa lista. 
"""

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
)

@tool(response_format="content_and_artifact")
def retriever(query: str): 
    """  
    Retrieve information related to a query.
    """
    retrieved_docs = vector_store.similarity_search(
        query = query, 
        k     = 2
    )
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\n" f"Content: {doc.page_content}") 
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

def query_or_respond(state: MessagesState):
    """
    Generate tool call for retrieval or respond.
    """
    llm_with_tools = llm.bind_tools([retriever])
    response = llm_with_tools.invoke(state["messages"])

    # MessagesState appends messages to state instead of overwriting
    return {"messages": [response]}

def generate(state: MessagesState):
    """
    Generate answer.
    """
    recent_tool_messages = []
    for message in reversed(state["messages"]):
        if message.type == "tool":
            recent_tool_messages.append(message)
        else:
            break
    tool_messages = recent_tool_messages[::-1]

    # Format into prompt
    docs_content = "\n\n".join(doc.content for doc in tool_messages)
    system_message_content = (
        f"{system_prompt}\n\n{docs_content}"
    )
    conversation_messages = [
        message
        for message in state["messages"]
        if message.type in ("human", "system")
        or (message.type == "ai" and not message.tool_calls)
    ]
    prompt = [SystemMessage(system_message_content)] + conversation_messages

    # Run
    response = llm.invoke(prompt)
    return {"messages": [response]}

tools = ToolNode([retriever])

graph_builder = StateGraph(MessagesState)
graph_builder.add_node(query_or_respond)
graph_builder.add_node(tools)
graph_builder.add_node(generate)

graph_builder.set_entry_point("query_or_respond")
graph_builder.add_conditional_edges(
    "query_or_respond",
    tools_condition,
    {END: END, "tools": "tools"},
)
graph_builder.add_edge("tools", "generate")
graph_builder.add_edge("generate", END)

memory = MemorySaver()

graph = graph_builder.compile(checkpointer=memory)
    

In [46]:
input_message   = "Hello, how are you ?"
input_message_2 = "What is about the theme for this article ?"
input_message_3 = "What a sayed for you in last time ?"
input_message_4 = "Do you know what is dark wave genre ?"

config = {"configurable": {"thread_id": "935"}}

In [42]:
for step in graph.stream(
    {"messages": [{"role": "user", "content": input_message}]},
    stream_mode="values",
    config=config
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Hello, how are you ?
================================== Ai Message ==================================

I'm doing well, thank you for asking! How about you?


In [47]:
for step in graph.stream(
    {"messages": [{"role": "user", "content": input_message_2}]},
    stream_mode="values",
    config=config
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is about the theme for this article ?
================================== Ai Message ==================================
Tool Calls:
  retriever (call_z62g)
 Call ID: call_z62g
  Args:
    query: theme of the article about AI and mental health
================================= Tool Message =================================
Name: retriever

Source: {'source': 'Review of AI and Mental Health.pdf', 'page': 9}
Content: w entries up to May 26, 2023. Weﬁne-tuned
our search strategy based on previous systematic reviews3,51,62 to
locate sources related to AI-based CAs for addressing mental
health problems or promoting mental well-being. The search was
limited to English-language publications. Complete lists of
datasets and search strategies are detailed in Supplementary
Table 7.
After removing duplicates, we screened all retrieved citations
and abstracts in two stages: title/abstract screening and full-text
re

In [32]:
for step in graph.stream(
    {"messages": [{"role": "user", "content": input_message_3}]},
    stream_mode="values",
    config=config
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What a sayed for you in last time ?
================================== Ai Message ==================================

I apologize, but I don't have any record of our previous conversations. I'm a large language model, I don't have personal memory, and each time you interact with me, it's a new conversation. If you'd like to continue a previous topic, feel free to bring it up again, and I'll do my best to help!


In [49]:
for step in graph.stream(
    {"messages": [{"role": "user", "content": input_message_3}]},
    stream_mode="values",
    config=config
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What a sayed for you in last time ?
================================== Ai Message ==================================

You asked me about an article, and I responded by calling a tool to retrieve information about the article. Then, you provided the tool's output, and I summarized the main points for you. After that, you asked about the theme of the article, and I again called a tool to retrieve the information. You provided the tool's output, and I told you that the theme of the article is the effectiveness of AI-based conversational agents in improving mental health and well-being.
